# ISLES'26 — End-to-End Colab Walkthrough

This notebook runs the full pipeline on the **ATLAS R2.1** (Anatomical Tracings of Lesions After Stroke) dataset — real T1w MRI brain scans with expert lesion annotations.

| | |
|---|---|
| Dataset | ATLAS R2.1 (655 subjects, chronic stroke, 1 mm isotropic T1w) |
| Raw data | 4.0 GB — skull-strip + normalise in pipeline |
| Preprocessed | 9.7 GB — MNI-registered, skull-stripped (**recommended**) |
| Source | [NITRC](https://www.nitrc.org/projects/atlas) / AWS S3 |

**Steps**
1. Install dependencies & clone repo
2. Download & extract ATLAS R2.1
3. Reorganise BIDS → pipeline raw format
4. Preprocess (Z-score normalise)
5. Create centre-stratified K-fold splits
6. Train SegResNet+FiLM (debug run or full)
7. Evaluate with stratified metrics
8. Ensemble inference on a held-out subject
9. Visualise predictions
10. Per-lesion metrics, postprocessing walkthrough, FiLM sanity check

> **Runtime**: GPU T4 recommended. Use **High-RAM** runtime for the full 655-subject dataset.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/isles2026-Ischemic-Stroke-Lesion-Segmentation/blob/main/notebooks/02_colab_e2e.ipynb)

## 1 · Install dependencies

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --depth 1 https://github.com/hashirama21/isles2026-Ischemic-Stroke-Lesion-Segmentation.git /content/isles26
    %cd /content/isles26
    !pip install -q -e '.[dev]'
    print('Installed.')
else:
    import os
    os.chdir('/Users/krohn/PycharmProjects/isles26')
    print('Local mode.')

In [ ]:
import torch
import pytorch_lightning as pl
import monai

print(f'PyTorch        {torch.__version__}')
print(f'Lightning      {pl.__version__}')
print(f'MONAI          {monai.__version__}')
print(f'CUDA available {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            {torch.cuda.get_device_name(0)}')

!df -h /content 2>/dev/null || df -h .

## 2 · Download ATLAS R2.1

Choose between **preprocessed** (MNI-registered, skull-stripped — recommended) and **raw** (4 GB, requires skull-stripping).

Both tarballs are hosted on the INDI AWS S3 bucket and are publicly accessible.

In [ ]:
from pathlib import Path

# Set to 'preprocessed' (9.7 GB, recommended) or 'raw' (4.0 GB)
ATLAS_VERSION = 'preprocessed'

ATLAS_URLS = {
    'raw':          'https://fcp-indi.s3.us-east-1.amazonaws.com/data/Projects/INDI/ATLAS/R2.1/atlas21_training_raw.tar.gz',
    'preprocessed': 'https://fcp-indi.s3.us-east-1.amazonaws.com/data/Projects/INDI/ATLAS/R2.1/atlas21_training_preprocessed.tar.gz',
}

ATLAS_DIR  = Path('data/atlas_raw')
RAW_DIR    = Path('data/raw')
TARBALL    = Path(f'atlas21_training_{ATLAS_VERSION}.tar.gz')

ATLAS_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

url = ATLAS_URLS[ATLAS_VERSION]

if not TARBALL.exists():
    print(f'Downloading {url} ...')
    !wget -q --show-progress -O "{TARBALL}" "{url}"
else:
    print(f'Tarball already present: {TARBALL}')

## 3 · Extract the tarball

In [ ]:
import tarfile

if not any(ATLAS_DIR.iterdir()):
    print(f'Extracting {TARBALL} → {ATLAS_DIR}/ ...')
    with tarfile.open(TARBALL) as tar:
        tar.extractall(ATLAS_DIR)
    print('Done.')
else:
    print('Already extracted.')

# Show top-level structure
tops = sorted(ATLAS_DIR.iterdir())[:5]
for p in tops:
    print(p)

## 4 · Inspect BIDS layout & discover subject pairs

ATLAS R2.1 follows BIDS:
```
sub-r{site}s{num}/ses-1/anat/
  sub-r001s001_ses-1_T1w.nii.gz
  sub-r001s001_ses-1_label-L_desc-T1lesion_mask.nii.gz
```
The site code inside the subject ID is used as the acquisition **centre**.

In [ ]:
import re

def find_atlas_pairs(root: Path) -> list[dict]:
    """Walk the BIDS tree and return (t1w_path, mask_path, subject_id, center) dicts."""
    pairs = []
    for t1w in sorted(root.rglob('*_T1w.nii.gz')):
        anat_dir = t1w.parent
        # Lesion mask: label-L_desc-T1lesion_mask or any *mask*.nii.gz in same dir
        masks = (
            sorted(anat_dir.glob('*label-L_desc-T1lesion_mask.nii.gz')) or
            sorted(anat_dir.glob('*_mask.nii.gz'))
        )
        if not masks:
            continue  # skip subjects without annotation
        subj_id = t1w.name.replace('_T1w.nii.gz', '').split('_ses-')[0]
        # Extract site code: sub-r001s001 → r001
        m = re.search(r'sub-(r\d+)s\d+', subj_id)
        center = m.group(1) if m else 'unknown'
        pairs.append({
            't1w':     t1w,
            'mask':    masks[0],
            'subj_id': subj_id.replace('sub-', ''),
            'center':  center,
        })
    return pairs


pairs = find_atlas_pairs(ATLAS_DIR)
print(f'Found {len(pairs)} annotated subjects')

centers = {p['center'] for p in pairs}
print(f'Acquisition centres ({len(centers)}): {sorted(centers)}')

# Preview one entry
p0 = pairs[0]
print(f'\nExample subject: {p0["subj_id"]}')
print(f'  T1w  : {p0["t1w"]}')
print(f'  Mask : {p0["mask"]}')
print(f'  Site : {p0["center"]}')

## 5 · Reorganise into pipeline raw format

Our `preprocess.py` expects:
```
data/raw/<subj_id>/
  <subj_id>_T1w.nii.gz
  <subj_id>_mask.nii.gz
  <subj_id>_meta.json   ← DAYS_POST_STROKE / CHRONICITY / CENTER
```

ATLAS is a **chronic** stroke dataset (all subjects ≥ 90 days post stroke). No exact `DAYS_POST_STROKE` is provided, so we use the conservative value of **180 days** and set `CHRONICITY = 2`.

Set `MAX_SUBJECTS` to limit the number of subjects for a quick demo (use `None` for the full dataset).

In [ ]:
import json
import shutil

MAX_SUBJECTS = 50  # set to None to use all 655 subjects

subset = pairs[:MAX_SUBJECTS] if MAX_SUBJECTS else pairs
print(f'Reorganising {len(subset)} subjects → {RAW_DIR}/')

for p in subset:
    sid = p['subj_id']
    dest = RAW_DIR / sid
    dest.mkdir(exist_ok=True)

    t1w_dst  = dest / f'{sid}_T1w.nii.gz'
    mask_dst = dest / f'{sid}_mask.nii.gz'
    meta_dst = dest / f'{sid}_meta.json'

    if not t1w_dst.exists():
        shutil.copy2(p['t1w'],  t1w_dst)
    if not mask_dst.exists():
        shutil.copy2(p['mask'], mask_dst)

    meta = {
        'DAYS_POST_STROKE': 180,
        'CHRONICITY': 2,
        'CENTER': p['center'],
    }
    with open(meta_dst, 'w') as f:
        json.dump(meta, f)

ready = sorted(RAW_DIR.glob('*/  *_T1w.nii.gz'))
print(f'Ready: {len(list(RAW_DIR.glob("*/*_T1w.nii.gz")))} T1w volumes in {RAW_DIR}/')

## 6 · Preprocessing

For the **preprocessed** ATLAS variant the volumes are already skull-stripped; the pipeline's intensity-threshold fallback (applied when HD-BET is absent) is a no-op on already-zeroed background voxels.
For the **raw** variant HD-BET will run automatically if installed, otherwise the same fallback applies.

In [ ]:
!python scripts/preprocess.py \
    data.raw_dir=data/raw \
    data.out_dir=data/processed \
    preprocess.n_jobs=4

In [ ]:
from pathlib import Path

processed = sorted(Path('data/processed').glob('*/metadata.json'))
print(f'{len(processed)} processed subjects')

with open(processed[0]) as f:
    print(json.dumps(json.load(f), indent=2))

## 7 · Lesion statistics across the dataset

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

volumes_ml = []
for meta_path in processed:
    with open(meta_path) as f:
        m = json.load(f)
    lbl_path = m.get('label_path', '')
    if lbl_path and Path(lbl_path).exists():
        nib_img = nib.load(lbl_path)
        zooms   = nib_img.header.get_zooms()[:3]
        vox_ml  = float(np.prod(zooms)) * 0.001
        vol_ml  = float(np.asarray(nib_img.dataobj).sum()) * vox_ml
        volumes_ml.append(vol_ml)

volumes_ml = np.array(volumes_ml)
print(f'Lesion volume  min={volumes_ml.min():.2f}  median={np.median(volumes_ml):.2f}  max={volumes_ml.max():.2f} mL')
print(f'No-lesion subjects: {(volumes_ml == 0).sum()}')
print(f'Small lesion (<1 mL): {((volumes_ml > 0) & (volumes_ml < 1)).sum()}')
print(f'Large lesion (>=1 mL): {(volumes_ml >= 1).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(volumes_ml[volumes_ml > 0], bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Lesion volume (mL)')
axes[0].set_ylabel('# subjects')
axes[0].set_title('Lesion volume distribution')
axes[1].hist(np.log1p(volumes_ml[volumes_ml > 0]), bins=40, color='darkorange', edgecolor='white')
axes[1].set_xlabel('log(1 + lesion volume)')
axes[1].set_title('Log-scale distribution')
plt.tight_layout()
plt.show()

## 8 · Create centre-stratified K-fold splits

In [ ]:
!python scripts/make_splits.py splits.n_folds=5 splits.test_ratio=0.10

In [ ]:
with open('data/splits/splits_5fold.json') as f:
    splits = json.load(f)

for fold, d in splits.items():
    val_centers = {s.get('center', '?') for s in d['val']}
    print(f"{fold}: train={len(d['train'])}  val={len(d['val'])}  test={len(d['test'])}  val_centers={sorted(val_centers)}")

## 9 · Weighted lesion sampler

The sampler over-weights small-lesion subjects (< 1 mL) at 3×, which is critical for the ATLAS dataset given its wide volume distribution.

In [ ]:
from src.data.sampler import build_lesion_sampler
from collections import Counter

fold0_train = splits['fold_0']['train']
sampler = build_lesion_sampler(fold0_train, small_lesion_threshold_ml=1.0)
weights = sampler.weights.numpy()

wt_counts = Counter(weights)
labels = {0.5: 'no lesion', 1.0: 'large (≥1 mL)', 3.0: 'small (<1 mL)'}
for w, cnt in sorted(wt_counts.items()):
    print(f'  weight {w:.1f}  ({labels.get(w, "?")}): {cnt} subjects')

plt.figure(figsize=(5, 3))
plt.bar([labels.get(k, str(k)) for k in sorted(wt_counts)],
        [wt_counts[k] for k in sorted(wt_counts)],
        color=['#d62728', '#1f77b4', '#2ca02c'])
plt.ylabel('# subjects')
plt.title('Sampling weight distribution — fold 0 train')
plt.tight_layout()
plt.show()

## 10 · Training

### 10a — Debug run (2 iterations, no checkpoint saved)
Verifies the full forward/backward loop end-to-end.

In [ ]:
!python scripts/train.py \
    experiment=segresnet_baseline \
    fold=0 \
    debug=true \
    data.num_workers=2 \
    wandb.mode=disabled

### 10b — Full training run (set `RUN_FULL_TRAIN = True`)

Trains fold 0 for 300 epochs with early stopping (patience 50). With a T4 GPU and 50 subjects, one epoch takes ~30 s.

In [ ]:
RUN_FULL_TRAIN = False

if RUN_FULL_TRAIN:
    !python scripts/train.py \
        experiment=segresnet_baseline \
        fold=0 \
        training.max_epochs=300 \
        data.num_workers=4 \
        wandb.mode=disabled
else:
    print('Set RUN_FULL_TRAIN=True to enable.')

## 11 · Evaluation with stratified metrics

In [ ]:
import glob

ckpts = sorted(glob.glob('outputs/checkpoints/fold0/*.ckpt'))
CKPT = ckpts[-1] if ckpts else None
print(f'Checkpoint: {CKPT}' if CKPT else 'No checkpoint (debug run does not save one).')

In [ ]:
if CKPT:
    !python scripts/evaluate.py \
        checkpoint="{CKPT}" \
        fold=0 \
        data.num_workers=2 \
        experiment=segresnet_baseline
else:
    print('Skipped — no checkpoint available.')

## 12 · Direct inference on a held-out subject

In [ ]:
import torch
import nibabel as nib
import numpy as np
from omegaconf import OmegaConf
from pathlib import Path
from src.inference.predict import ensemble_predict

# Use first test-set subject
test_subjects = splits['fold_0']['test']
test_meta = test_subjects[0]
print(f"Subject : {test_meta['subject_id']}")
print(f"Center  : {test_meta.get('center', '?')}")

img_nib = nib.load(test_meta['image_path'])
data_np = np.asarray(img_nib.dataobj, dtype=np.float32)

image_tensor = torch.from_numpy(data_np).unsqueeze(0).unsqueeze(0)  # [1,1,D,H,W]
meta_tensor  = torch.tensor(
    [min(float(test_meta.get('days_post_stroke', 180)) / 365.0, 1.0),
     float(test_meta.get('chronicity', 2))],
    dtype=torch.float32,
).unsqueeze(0)  # [1,2]

print(f'Image   : {image_tensor.shape}')
print(f'Meta    : {meta_tensor}')

In [ ]:
postprocess_cfg = OmegaConf.create({
    'postprocess': {
        'min_lesion_volume_ml': 0.05,
        'threshold_acute': 0.45,
        'threshold_chronic': 0.35,
    }
})

model_cfg = OmegaConf.create({
    '_target_': 'src.models.segresnet.SegResNetFiLM',
    'in_channels': 1,
    'out_channels': 2,
    'init_filters': 32,
    'blocks_down': [1, 2, 2, 4],
    'blocks_up': [1, 1, 1],
    'dropout_prob': 0.2,
    'meta_dim': 2,
    'film_hidden_dim': 64,
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
spacing = tuple(float(z) for z in img_nib.header.get_zooms()[:3])

if CKPT:
    pred = ensemble_predict(
        checkpoint_paths=[Path(CKPT)],
        model_cfgs=[model_cfg],
        image=image_tensor,
        metadata=meta_tensor,
        roi_size=[128, 128, 128],
        use_tta=False,
        device=device,
        postprocess_cfg=postprocess_cfg,
        voxel_spacing_mm=spacing,
    )
    print(f'Prediction shape: {pred.shape}  unique: {np.unique(pred)}')
    vox_ml = float(np.prod(spacing)) * 0.001
    print(f'Predicted lesion volume: {pred.sum() * vox_ml:.2f} mL')
else:
    print('No checkpoint — using a placeholder prediction for visualisation.')
    pred = np.zeros(data_np.shape, dtype=np.uint8)

## 13 · Visualisation

Three axial slices centred around the ground-truth lesion (or the image midpoint if no lesion is present).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

gt_nib = nib.load(test_meta['label_path'])
gt     = np.asarray(gt_nib.dataobj, dtype=np.uint8)

# Centre slices on the lesion centroid (or image midpoint)
if gt.any():
    cz = int(np.argwhere(gt).mean(axis=0)[0])
else:
    cz = data_np.shape[0] // 2

D = data_np.shape[0]
slices = sorted({max(0, cz - 10), cz, min(D - 1, cz + 10)})

fig, axes = plt.subplots(3, len(slices), figsize=(5 * len(slices), 12))
if len(slices) == 1:
    axes = axes[:, None]  # keep 2-D indexing

for col, sl in enumerate(slices):
    t1_sl   = data_np[sl]
    gt_sl   = gt[sl]
    pred_sl = pred[sl]

    axes[0, col].imshow(t1_sl, cmap='gray')
    axes[0, col].set_title(f'T1w  z={sl}')
    axes[0, col].axis('off')

    axes[1, col].imshow(t1_sl, cmap='gray')
    if gt_sl.any():
        axes[1, col].imshow(np.ma.masked_where(gt_sl == 0, gt_sl),
                            cmap='Reds', alpha=0.65, vmin=0, vmax=1)
    axes[1, col].set_title(f'Ground truth  z={sl}')
    axes[1, col].axis('off')

    axes[2, col].imshow(t1_sl, cmap='gray')
    if pred_sl.any():
        axes[2, col].imshow(np.ma.masked_where(pred_sl == 0, pred_sl),
                            cmap='Blues', alpha=0.65, vmin=0, vmax=1)
    axes[2, col].set_title(f'Prediction  z={sl}')
    axes[2, col].axis('off')

red_patch  = mpatches.Patch(color='red',  alpha=0.65, label='Ground truth')
blue_patch = mpatches.Patch(color='blue', alpha=0.65, label='Prediction')
fig.legend(handles=[red_patch, blue_patch], loc='lower center', ncol=2, fontsize=12)
fig.suptitle(
    f"Subject: {test_meta['subject_id']}  |  Centre: {test_meta.get('center', '?')}  |  ATLAS R2.1 (chronic)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## 14 · Per-lesion metrics

In [ ]:
from src.evaluation.metrics import compute_metrics

metrics = compute_metrics(pred, gt, voxel_spacing_mm=spacing)

print(f"Dice score        : {metrics['dice']:.4f}")
print(f"Precision         : {metrics['precision']:.4f}")
print(f"Recall            : {metrics['recall']:.4f}")
if metrics.get('hd95') is not None:
    print(f"HD95 (mm)         : {metrics['hd95']:.2f}")
    print(f"ASSD (mm)         : {metrics['assd']:.2f}")
print(f"Pred volume (mL)  : {metrics['pred_volume_ml']:.4f}")
print(f"GT   volume (mL)  : {metrics['gt_volume_ml']:.4f}")

## 15 · Postprocessing walkthrough

In [ ]:
from src.inference.postprocess import remove_small_components, fill_holes, adaptive_threshold

# Simulate a probability map from the real image region
prob_map = np.zeros(data_np.shape, dtype=np.float32)
if gt.any():
    prob_map[gt == 1] = 0.80          # true lesion region
    # add a spurious tiny blob
    prob_map[cz, 10:12, 10:12] = 0.50
else:
    prob_map[cz, 40:55, 40:55] = 0.80

days   = float(test_meta.get('days_post_stroke', 180))
binary = adaptive_threshold(prob_map, days_post_stroke=days)
print(f'After threshold    : {binary.sum()} voxels')

vox_ml  = float(np.prod(spacing)) * 0.001
min_vox = 0.05 / vox_ml
cleaned = remove_small_components(binary, min_volume_vox=min_vox)
print(f'After CC filtering : {cleaned.sum()} voxels')

filled = fill_holes(cleaned)
print(f'After hole fill    : {filled.sum()} voxels')

## 16 · FiLM conditioning sanity check

Verify that the model produces different outputs for acute vs. chronic metadata.

In [ ]:
from hydra import initialize_config_dir, compose
from hydra.utils import instantiate
import os

config_dir = str(Path(os.getcwd()) / 'configs')

with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name='config', overrides=['experiment=segresnet_baseline'])

model = instantiate(cfg.model).eval()

dummy_img    = torch.zeros(1, 1, 64, 64, 64)
meta_acute   = torch.tensor([[0.01, 0.0]])   # ~4 days, acute
meta_chronic = torch.tensor([[0.49, 2.0]])   # 180 days, chronic

with torch.no_grad():
    out_acute   = model(dummy_img, meta_acute)
    out_chronic = model(dummy_img, meta_chronic)

diff = (out_acute - out_chronic).abs().mean().item()
print(f'Mean |output| diff (acute vs chronic): {diff:.6f}')
print('FiLM conditioning is active.' if diff > 1e-6 else 'WARNING: FiLM has no effect!')

## 17 · Next steps

| Step | Command |
|------|---------|
| Full 5-fold training | `python scripts/train.py fold={0..4} training.max_epochs=500` |
| NNUNet default model | `python scripts/train.py fold=0 training.max_epochs=500` |
| Evaluate fold | `python scripts/evaluate.py checkpoint=outputs/checkpoints/fold0/best.ckpt fold=0` |
| Ensemble + TTA | `python scripts/ensemble.py ensemble.use_tta=true` |
| Build Docker image | `bash docker/build.sh` |
| Run tests | `pytest tests/ -v` |

**Using the full ATLAS dataset**: set `MAX_SUBJECTS = None` in section 5 and re-run from that cell onward.

**Official ISLES'26 data**: replace the ATLAS download with the ISLES'26 dataset from [isles-challenge.org](https://www.isles-challenge.org). The pipeline is identical; only the metadata keys (`DAYS_POST_STROKE`, `CHRONICITY`) will be populated from the challenge JSON files.